## Scroll 처리
- SCROLL_COUNT : 스크롤 횟수 → **화면에 로드되는 영상 수**가 결정됨
- MAX_PER_KEYWORD : 로드된 영상 중 **저장할 최대 개수**

In [1]:
import time
from urllib.parse import quote

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

KEYWORD = "경제 뉴스"
MAX_PER_KEYWORD = 5 # 영상 저장 최대 개수 
SCROLL_COUNT = 1 # 스크롤 횟수

driver = webdriver.Chrome()
# quote(KEYWORD) : 한글이나 공백이 들어간 검색어를 URL에 쓸 수 있게 퍼센트 인코딩 하는 함수 
driver.get(f"https://www.youtube.com/results?search_query={quote(KEYWORD)}")

WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "ytd-video-renderer"))
)

<selenium.webdriver.remote.webelement.WebElement (session="d09cdfeb4fdc75b82ef717b72ca81f79", element="f.FF92E3DFBDC34CE40C02E215DF545471.d.7DFA2466FA4BE17E82767DEFC4D76657.e.31")>

In [3]:
# SCROLL_COUNT만큼 스크롤 -> 영상이 더 로드됨
    # execute_script : 브라우저에서 자바스크립트 실행 
    # window.scrollTo(0, 페이지 전체 높이) : 맨 아래로 스크롤 한 후 바닥에 닿으면 YouTube가 영상을 추가로 불러옴
    # 1 SCROLL_COUNT : 5~20개 증가
for i in range(SCROLL_COUNT):
    driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
    time.sleep(1.5)  # 스크롤되는 시간 확보를 위한 1.5초 

rows = driver.find_elements(By.CSS_SELECTOR, "ytd-video-renderer")
print(f"로드된 영상 수: {len(rows)}개  (SCROLL_COUNT={SCROLL_COUNT})")

로드된 영상 수: 46개  (SCROLL_COUNT=1)


In [ ]:
# MAX_PER_KEYWORD만큼만 저장
result = []
for row in rows:
    if len(result) >= MAX_PER_KEYWORD:
        break

    title_el = row.find_element(By.CSS_SELECTOR, "a#video-title")
    result.append({
        "keyword": KEYWORD,
        "title": title_el.get_attribute("title"),
        "url": title_el.get_attribute("href")
    })

print(f"수집한 영상 수: {len(result)}개  (MAX_PER_KEYWORD={MAX_PER_KEYWORD})")

driver.quit()

수집한 영상 수: 5개  (MAX_PER_KEYWORD=5)


In [5]:
df = pd.DataFrame(result)
df

,keyword,title,url
0,경제 뉴스,[손경제] SK하이닉스 특별배당 | 주식-부동산 | 한전 사채 | 가상자산 과세 |...,https://www.youtube.com/watch?v=OwCMZq49ERA&pp...
1,경제 뉴스,[18탄 / 경제 모아보기] 아는 척하기 딱 좋은 재미있는 경제 뉴스 10가지｜크랩,https://www.youtube.com/watch?v=jmF_3mrTTPk&pp...
2,경제 뉴스,"’막판 역전극’ 코스피, 5500선 사수할까?ㅣ롤러코스터 장세 담아둘 TOP4 l ...",https://www.youtube.com/watch?v=bmVzMmfecbE&pp...
3,경제 뉴스,[이슈체크] 세계 경제 흔들 복병 ‘프랑스 국채’…지금 '이 변수'를 조심해야 하는 이유,https://www.youtube.com/watch?v=-vjFb_mwD6U&pp...
4,경제 뉴스,🔴[생방송] 삼전닉스 추가 주주환원 주목...이번 주 반도체 주가 향방은? | 마...,https://www.youtube.com/watch?v=uP0-t5l6Ylo&pp...


## 금융 데이터 수집(Youtube)
* ["경제 뉴스", "주식 분석", "ETF 추천", "재미로 보는 주식"]

In [ ]:
# title, url, view 수, hash tab, 게시일

## 1. 라이브러리 & 설정

In [15]:
import re
import time
from urllib.parse import quote

import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ===================== 설정 =====================
KEYWORDS = ["경제 뉴스", "주식 분석", "ETF 추천", "재미로 보는 주식"]
MAX_PER_KEYWORD = 10      # 키워드당 수집할 영상 수
SCROLL_COUNT = 3          # 스크롤 횟수 (많을수록 더 많은 영상 로드)
OUTPUT_FILE = "youtube_result.csv"
# ================================================

## 2. 함수 정의

In [16]:
def get_text(parent, selector):
    try:
        el = parent.find_element(By.CSS_SELECTOR, selector)
        return (el.get_attribute("textContent") or "").strip()
    except Exception:
        return ""

def get_attr(parent, selector, attr):
    try:
        el = parent.find_element(By.CSS_SELECTOR, selector)
        return (el.get_attribute(attr) or "").strip()
    except Exception:
        return ""

## 3. 브라우저 실행

In [17]:
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--lang=ko-KR")
# options.add_argument("--headless=new")   # 창 없이 실행하려면 주석 해제
driver = webdriver.Chrome(options=options)

## 4. 크롤링

* test code

In [4]:
keyword = "경제 뉴스"
driver.get(f"https://www.youtube.com/results?search_query={quote(keyword)}")

# 검색 결과 로딩 대기
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.CSS_SELECTOR, "ytd-video-renderer"))
)

# 스크롤해서 영상 더 불러오기
for _ in range(SCROLL_COUNT):
    driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
    time.sleep(1.5)

In [12]:
rows = driver.find_elements(By.CSS_SELECTOR, "ytd-video-renderer")
count = 0

for row in rows:
    if count >= MAX_PER_KEYWORD:
        break

    title = get_text(row, "a#video-title")
    url = get_attr(row, "a#video-title", "href")

    # 제목/URL이 없는 행 제외
    if not title or not url:
        continue

    # 채널명, 채널 URL
    channel = get_text(row, "yt-formatted-string #text a") or get_text(row, "yt-formatted-string a")
    channel_url = get_attr(row, "yt-formatted-string a", "href")

    # 조회수 / 업로드 시점 (metadata-line 안의 span 2개)
    meta = row.find_elements(By.CSS_SELECTOR, "#metadata-line span.inline-metadata-item")
    meta_texts = [(m.get_attribute("textContent") or "").strip() for m in meta]

    # "조회수 7.2만"
    views_text = meta_texts[0].replace("조회수", "").replace("회", "").strip()
    uploaded = meta_texts[1]

    # 영상 길이
    duration = get_text(row, "ytd-thumbnail-overlay-time-status-renderer #text")

    # 영상 설명(요약)
    snippet = get_text(row, ".metadata-snippet-text")


    print(snippet)
    print(duration)
    print(views_text)
    print(uploaded)
    
    print(meta)
    print(meta_texts)

    print(channel)
    print(channel_url)
    
    print(title)
    print(url)

    break

금값 #금시세 #안전자산 #국제유가 #금리인상 #원자재 #금투자 #증시전망 #금융시장 #경제뉴스 * 본 방송은 10월 26일에 촬영 ...
2:09
7.2만
1일 전
[<selenium.webdriver.remote.webelement.WebElement (session="2b54e19cb79da71de2bde6ab8a4d090b", element="f.CB9A8F4632DBE78627182FBC562DF12F.d.941C7E42EAD2F69C231A73AECB6BCECF.e.2403")>, <selenium.webdriver.remote.webelement.WebElement (session="2b54e19cb79da71de2bde6ab8a4d090b", element="f.CB9A8F4632DBE78627182FBC562DF12F.d.941C7E42EAD2F69C231A73AECB6BCECF.e.2404")>]
['7.2만', '1일 전']
서울경제TV
https://www.youtube.com/@SeoulEconomyTV
금, 안판 엄마들 돌연 날벼락…불과 9개월 만에 ‘36.5%’ 와르르
https://www.youtube.com/watch?v=2nB6VW3CdpI&pp=ygUN6rK97KCcIOuJtOyKpNIHCQkwDAGHKiGM7w%3D%3D


# 실 코드

In [18]:
result = []

for keyword in KEYWORDS:
    print(f"[검색] {keyword}")
    driver.get(f"https://www.youtube.com/results?search_query={quote(keyword)}")

    # 검색 결과 로딩 대기
    WebDriverWait(driver, 10).until(
        EC.presence_of_element_located((By.CSS_SELECTOR, "ytd-video-renderer"))
    )

    # 스크롤해서 영상 더 불러오기
    for _ in range(SCROLL_COUNT):
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        time.sleep(1.5)

    rows = driver.find_elements(By.CSS_SELECTOR, "ytd-video-renderer")
    count = 0

    for row in rows:
        if count >= MAX_PER_KEYWORD:
            break

        title = get_text(row, "a#video-title")
        url = get_attr(row, "a#video-title", "href")

        # 제목/URL이 없는 행 제외
        if not title or not url:
            continue

        # 채널명, 채널 URL
        channel = get_text(row, "ytd-channel-name #text a") or get_text(row, "ytd-channel-name a")
        channel_url = get_attr(row, "ytd-channel-name a", "href")

        # 조회수 / 업로드 시점 (metadata-line 안의 span 2개)
        meta = row.find_elements(By.CSS_SELECTOR, "#metadata-line span.inline-metadata-item")
        meta_texts = [(m.get_attribute("textContent") or "").strip() for m in meta]

        # 조회수가 없는 행(라이브 예정 등) 제외
        if len(meta_texts) < 2:
            continue

        views_text = meta_texts[0].replace("조회수", "").replace("회", "").strip()
        uploaded = meta_texts[1]

        # 영상 길이
        duration = get_text(row, "ytd-thumbnail-overlay-time-status-renderer #text")
        if not duration:
            duration = get_text(row, "badge-shape .badge-shape-wiz__text")

        # 영상 설명(요약)
        snippet = get_text(row, ".metadata-snippet-text")

        result.append({
            "keyword": keyword,
            "title": title,
            "channel": channel,
            "channel_url": channel_url,
            "views_text": views_text,
            "views": views_text,
            "uploaded": uploaded,
            "duration": duration,
            "snippet": snippet,
            "url": url
        })
        count += 1

    print(f"  -> {count}개 수집")

print(f"총 {len(result)}개 수집 완료")

[검색] 경제 뉴스
  -> 10개 수집
[검색] 주식 분석
  -> 10개 수집
[검색] ETF 추천
  -> 10개 수집
[검색] 재미로 보는 주식
  -> 10개 수집
총 40개 수집 완료


## 5. 브라우저 종료

In [19]:
driver.quit()

## 6. DataFrame 생성 & CSV 저장

In [20]:
# DataFrame 생성 (중복 제거 + 조회수 높은 순 정렬)
df = pd.DataFrame(result)
df = df.drop_duplicates(subset="url")
df = df.sort_values("views", ascending=False).reset_index(drop=True)

# CSV 저장
df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(f"CSV 저장 완료: {OUTPUT_FILE}")
df

CSV 저장 완료: youtube_result.csv


,keyword,title,channel,channel_url,views_text,views,uploaded,duration,snippet,url
0,경제 뉴스,[손경제] SK하이닉스 특별배당 | 주식-부동산 | 한전 사채 | 가상자산 과세 |...,손에잡히는경제,https://www.youtube.com/@%EC%86%90%EA%B2%BD%EC...,9만,9만,스트리밍 시간: 1일 전,23:43,... 코인 #경제뉴스 #손경제 #이진우기자 ※진행: 이진우 기자 [깊이 있는 경제...,https://www.youtube.com/watch?v=OwCMZq49ERA&pp...
1,경제 뉴스,주식에 물려있다면 이 날까지 희망을 가져봅시다! (문홍철 & 이진우) | 인포맥스라...,연합뉴스경제TV,https://www.youtube.com/@%EC%97%B0%ED%95%A9%EB...,94만,94만,3일 전,42:31,출연: 문홍철 DB증권 자산전략팀장 ‍♂️진행: 이진우 GFM투자연구소 소장 광고/...,https://www.youtube.com/watch?v=Q-Dq99bMQos&pp...
2,경제 뉴스,고점에서 4조원 물렸다…ELS 투자자 '멘붕' / 한국경제TV뉴스,한국경제TV뉴스,https://www.youtube.com/@hkwowtvnews,8.9만,8.9만,5일 전,3:00,"국내 증시 변동성이 높아지자 주가연계증권, ELS 투자자들의 불안도 커지고 있습니다...",https://www.youtube.com/watch?v=U8Ber6Fr84k&pp...
3,주식 분석,"미국증시와 한국증시 시황과전략(AI반도체,수급,환율,유가,금리,섹터별분석과논평)",TV주식의 강해,https://www.youtube.com/@harry91daum,8.5천,8.5천,5시간 전,14:38,"주식시장에서 맺은 소중한 인연, 평생 함께 하겠습니다] (초보자를 위한 알기쉬운 코...",https://www.youtube.com/watch?v=CVte9Jzwo1A&pp...
4,경제 뉴스,"이벤트로 꽉 채운 일주일...우리는 어떻게 대응해야 할까 (김민수, 민재기, 김영롱...",연합뉴스경제TV,https://www.youtube.com/@%EC%97%B0%ED%95%A9%EB...,8.2만,8.2만,13시간 전,42:15,"출연: 김민수 레몬리서치 대표, 민재기 KB증권 PRIME CLUB 팀장 ‍♂️진행...",https://www.youtube.com/watch?v=EU4FwyDWdNM&pp...
5,경제 뉴스,"금, 안판 엄마들 돌연 날벼락…불과 9개월 만에 ‘36.5%’ 와르르",서울경제TV,https://www.youtube.com/@SeoulEconomyTV,7.4만,7.4만,1일 전,2:09,금값 #금시세 #안전자산 #국제유가 #금리인상 #원자재 #금투자 #증시전망 #금융시...,https://www.youtube.com/watch?v=2nB6VW3CdpI&pp...
6,재미로 보는 주식,코스피는 지금 어디까지 왔을까?｜엘리어트 파동으로 보는 코스피의 현재 위치,재미로 보는 내주식,https://www.youtube.com/@%EB%82%B4%EC%A3%BC%EC...,65,65,3주 전,3:17,"코스피의 움직임을 엘리어트 파동 관점에서 살펴봅니다. 상승 1파부터 5파, 그리고 ...",https://www.youtube.com/watch?v=_eZWn871AxI&pp...
7,ETF 추천,"광고❌ ""8~10% 수익률이라고?"" ETF 이 영상으로 졸업하세요 (ETF 투자 총정리)",김짠부,https://www.youtube.com/@zzanboo,60만,60만,1년 전,9:48,어렵게 느껴졌던 ETF 이제 졸업 완료..?! 오늘 영상 보고 댓글 남겨준 찐짠이들...,https://www.youtube.com/watch?v=3K8Qy5u5KHE&pp...
8,주식 분석,"이번 주 삼성전자·SK하이닉스 실적 발표, 미국 금리까지 중요합니다",미국 주식 채널 - 에디 Eddie,https://www.youtube.com/@%EB%AF%B8%EA%B5%AD%EC...,6.5만,6.5만,18시간 전,8:14,하락의 끝에 대한 신호를 가장 빠르게 알 수 있는 - 에디의 텔레그렘 가입: ...,https://www.youtube.com/watch?v=ZtcSnITJpXU&t=...
9,ETF 추천,ETF는 언제 팔아야 할까? 존리가 ETF를 추천하는 이유 | 존리의 머니스쿨,존리의 부자학교,https://www.youtube.com/@%EC%A1%B4%EB%A6%AC%EC...,59만,59만,1개월 전,14:29,매일 존리와 소통하는 머니스쿨 멤버십 https://abr.ge/yblimsc 존리...,https://www.youtube.com/watch?v=Oe4WizYWqNw&pp...
